In [11]:
import sys
# Install the Hugging Face integration
!{sys.executable} -m pip install langchain-huggingface 

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import os
import pandas as pd
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR API KEY"

print("Hugging Face libraries loaded successfully.")

Hugging Face libraries loaded successfully.


In [24]:
# Dictionary of models: "Friendly Name": "Repo ID"
models_to_test = {
    "Zephyr (Mistral Base)": "HuggingFaceH4/zephyr-7b-beta",
    "Google (Gemma 2 9b)": "google/gemma-2-9b-it",
    "Alibaba (Qwen 2.5 7B)": "Qwen/Qwen2.5-7B-Instruct"
}
print(f"Prepared to test: {list(models_to_test.keys())}")

Prepared to test: ['Zephyr (Mistral Base)', 'Google (Gemma 2 9b)', 'Alibaba (Qwen 2.5 7B)']


In [14]:
input_bullets = [
    "Scheduled appointments, managed emails and calls for senior leadership.",
    "Provided end-to-end event logistics and meeting support.",
    "Familiarity with audit techniques, processes, and tools.",
    "Proficiency in using GST portals for registration, payment, and return filing.",
    "Extracted, cleaned & integrated multi-source datasets (Insurance Records, Customer Feedback); performed data profiling & executed sentiment analysis via Power Query to deliver customer-centric insights.",
    "Designed & deployed dynamic dashboards using Bar, Line, Ribbon, Donut & Matrix visualizations; implemented Role-Based Security (RLS) & scheduled automated data refresh for real-time, secure reporting."
]

In [32]:
# A single, strong instruction for all models
master_prompt = ChatPromptTemplate.from_template(
    """
    You are an expert Resume Writer. 
    Rewrite the following resume bullet point to be professional, impactful, and metric-driven.
    
    Constraints:
    1. Start with a strong action verb (e.g., Orchestrated, Engineered).
    2. Be concise (approx 10-15 words).
    3. Do NOT include any intro/outro text (like "Here is the rewrite"). ONLY output the new bullet.
    4. Make sure there is no other language except english
    Original Bullet: {input_text}
    
    Rewritten Bullet:
    """
)

In [33]:
results = []

print("Starting Multi-Model Benchmark (Hugging Face Chat Mode)...")

for bullet in input_bullets:
    row_data = {"Original Input": bullet}
    
    for friendly_name, repo_id in models_to_test.items():
        try:
            # 1. Initialize Endpoint 
            llm = HuggingFaceEndpoint(
                repo_id=repo_id,
                max_new_tokens=100,
                temperature=0.3,
                repetition_penalty=1.1
            )
            
            # 2. WRAP in ChatHuggingFace
            chat_model = ChatHuggingFace(llm=llm)
            
            # 3. Create chain using the CHAT MODEL, not the raw llm
            chain = master_prompt | chat_model | StrOutputParser()
            
            # 4. Run chain
            response = chain.invoke({"input_text": bullet})
            
            # Clean up response (sometimes models repeat the prompt)
            clean_response = response.replace(bullet, "").strip()
            row_data[friendly_name] = clean_response
            
        except Exception as e:
            # If one model fails, we log it but keep going
            row_data[friendly_name] = f"Error: {str(e)[:100]}..." 
            
    results.append(row_data)

print("Benchmarking complete!")

Starting Multi-Model Benchmark (Hugging Face Chat Mode)...
Benchmarking complete!


In [34]:
# Create DataFrame
df = pd.DataFrame(results)

# Configure pandas to show full text
pd.set_option('display.max_colwidth', None)

# Display result 
display(df)


,Original Input,Zephyr (Mistral Base),Google (Gemma 2 9b),Alibaba (Qwen 2.5 7B)
0,"Scheduled appointments, managed emails and calls for senior leadership.","Orchestrated efficient appointment scheduling and effectively managed email and call communications for senior leadership, resulting in a 35% increase in response time and a 20% boost in meeting attendance.",Streamlined communication and calendar management for senior leadership.,"Engineered efficient scheduling, managing over 50 senior leadership appointments, emails, and calls."
1,Provided end-to-end event logistics and meeting support.,"Orchestrated comprehensive event logistics and meeting support, resulting in a 30% increase in attendee satisfaction ratings and a 25% reduction in operational costs.",Engineered seamless event logistics and meeting support for [Number] attendees.,"Engineered comprehensive event logistics, enhancing meeting support by 30%."
2,"Familiarity with audit techniques, processes, and tools.",Orchestrated a 25% reduction in audit cycle time through implementation of streamlined processes and utilization of advanced auditing software. Demonstrated proficiency in various audit techniques and tools.,Leveraged audit techniques to streamline processes and enhance efficiency.,"Engineered audit processes, achieving 95% compliance with regulatory standards."
3,"Proficiency in using GST portals for registration, payment, and return filing.",Orchestrated efficient processing of 150 GST registrations and submitted accurate returns with 100% compliance to regulatory standards through proficient utilization of GST portals for registration and filing purposes.,"Streamlined GST compliance by managing registration, payments, and returns.","Engineered efficient GST portal processes for registration, payment, and return filing, enhancing accuracy by 25%."
4,"Extracted, cleaned & integrated multi-source datasets (Insurance Records, Customer Feedback); performed data profiling & executed sentiment analysis via Power Query to deliver customer-centric insights.","Orchestrated extraction, cleaning, and integration of multi-source insurance records and customer feedback data through Power Query, followed by data profiling and sentiment analysis to deliver customer-centric insights. Resulted in a 25% increase in customer satisfaction ratings.",Engineered customer-centric insights by integrating and analyzing multi-source datasets.,"Engineered multi-source datasets (Insurance Records, Customer Feedback) for data profiling and sentiment analysis, delivering customer-centric insights."
5,"Designed & deployed dynamic dashboards using Bar, Line, Ribbon, Donut & Matrix visualizations; implemented Role-Based Security (RLS) & scheduled automated data refresh for real-time, secure reporting.","Orchestrated creation and implementation of dynamic dashboards utilizing Bar, Line, Ribbon, and Matrix visualizations while integrating Role-Based Security (RLS) and automated data refresh for real-time, secure reporting. Metrics: Increased report consumption by 35% through enhanced visual analytics and streamlined access controls.","Engineered dynamic dashboards with 5+ visualization types, implementing RLS and automated data refreshes for real-time reporting.","Engineered dynamic dashboards with Bar, Line, Ribbon, Donut, and Matrix visualizations; implemented Role-Based Security and automated real-time data refresh."
